# Notebook 4 — Credit Risk Scoring
**BankGuard | Microsoft Fabric**

## What does this notebook do?

We assign a **risk score (0–100)** to each customer based on simple, understandable rules.

This is called **rule-based scoring** — it's a great starting point before you move to machine learning. Banks actually use this approach for many decisions because it's **transparent and explainable**.

## How does the score work?

We look at each customer and add up points based on warning signs:

| Warning Sign | Points Added |
|---|---|
| Loan overdue 31–90 days | +20 |
| Loan overdue 91+ days (NPA) | +40 |
| High KYC non-compliance | +15 |
| Multiple declined transactions | +10 |
| Low transaction activity (possible dormancy) | +10 |
| HIGH risk category in source system | +15 |

**Score interpretation:**
- 0–25 → Low Risk 🟢
- 26–50 → Medium Risk 🟡
- 51–75 → High Risk 🟠
- 76–100 → Very High Risk 🔴

**Run Notebooks 1, 2, 3 before this one.**

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType
import datetime

SCORE_DATE = datetime.date.today().strftime('%Y-%m-%d')

# Read the tables we need
silver_cust  = spark.read.table('silver_customers')
silver_loans = spark.read.table('silver_loans')
silver_txn   = spark.read.table('silver_transactions')

print('Tables loaded ✅')
print(f'Score date: {SCORE_DATE}')

## Step 1: Prepare Loan Features

For each customer, what does their loan history look like?

In [ ]:
# Summarize loan status per customer
loan_features = (
    silver_loans
    .groupBy('customer_id')
    .agg(
        # Maximum DPD across all loans
        # If a customer has 3 loans: dpd=[0, 45, 0] → max_dpd = 45
        F.max('dpd').alias('max_dpd'),

        # Number of loans that are NPA
        F.sum(F.col('is_npa').cast('int')).alias('npa_loan_count'),

        # Total outstanding loan amount
        F.sum('outstanding_amount').alias('total_outstanding'),

        # Number of loans
        F.count('loan_id').alias('loan_count'),
    )
)

print('Loan features computed:')
loan_features.show(5)

## Step 2: Prepare Transaction Features

For each customer, how is their recent transaction behaviour?

In [ ]:
# Look at last 30 days of transaction behaviour
# F.date_sub(F.current_date(), 30) means '30 days ago from today'

recent_txn_features = (
    silver_txn
    .filter(F.col('transaction_date') >= F.date_sub(F.current_date(), 30))
    .groupBy('customer_id')
    .agg(
        # How active is this customer?
        F.count('transaction_id').alias('txn_count_30d'),

        # How many transactions were declined? (sign of financial stress)
        F.sum(
            F.when(F.col('status') == 'DECLINED', 1).otherwise(0)
        ).alias('declined_count_30d'),
    )
)

print('Transaction features computed:')
recent_txn_features.show(5)

## Step 3: Combine Everything

Join all features onto the customer table.

In [ ]:
# Join customer profile + loan features + transaction features
df_scoring = (
    silver_cust
    .join(loan_features,        on='customer_id', how='left')
    .join(recent_txn_features,  on='customer_id', how='left')
    # Replace nulls with 0 for customers with no loans or no recent transactions
    .fillna({
        'max_dpd'           : 0,
        'npa_loan_count'    : 0,
        'total_outstanding' : 0,
        'loan_count'        : 0,
        'txn_count_30d'     : 0,
        'declined_count_30d': 0,
    })
)

print(f'Scoring dataset: {df_scoring.count():,} customers')

## Step 4: Calculate Risk Score

This is the core scoring logic. Each rule adds points to the score.

In [ ]:
# ── Individual Rule Scores ─────────────────────────────────────────────────
# Each rule gives a score between 0 and a max value
# We add them all together to get the final score

df_scored = df_scoring.withColumns({

    # RULE 1: DPD Score — based on how overdue the loan is
    # 0 DPD = 0 points, 31-90 days = 20 points, 91+ days = 40 points
    'score_dpd':
        F.when(F.col('max_dpd') >= 91, 40)
         .when(F.col('max_dpd') >= 31, 20)
         .when(F.col('max_dpd') >= 1,   5)
         .otherwise(0).cast(IntegerType()),

    # RULE 2: NPA Score — customer with any NPA loan is high risk
    'score_npa':
        F.when(F.col('npa_loan_count') >= 2, 20)   # Multiple NPAs = very bad
         .when(F.col('npa_loan_count') == 1, 10)   # One NPA = bad
         .otherwise(0).cast(IntegerType()),

    # RULE 3: KYC Score — incomplete KYC is a risk flag
    'score_kyc':
        F.when(F.col('is_kyc_done') == False, 15)
         .otherwise(0).cast(IntegerType()),

    # RULE 4: Declined Transactions — many declines suggest cash flow issues
    'score_declines':
        F.when(F.col('declined_count_30d') >= 5, 15)
         .when(F.col('declined_count_30d') >= 3, 10)
         .otherwise(0).cast(IntegerType()),

    # RULE 5: Inactivity — very low transaction activity can mean account issues
    'score_inactivity':
        F.when(F.col('txn_count_30d') == 0, 10)    # No transactions at all
         .when(F.col('txn_count_30d') <= 2,  5)    # Very low activity
         .otherwise(0).cast(IntegerType()),

    # RULE 6: Source System Risk Category
    'score_risk_category':
        F.when(F.col('risk_category') == 'HIGH',   15)
         .when(F.col('risk_category') == 'MEDIUM',  5)
         .otherwise(0).cast(IntegerType()),
})

# ── Total Score ────────────────────────────────────────────────────────────
# Add all individual rule scores together
# cap at 100 using F.least(total, 100)
df_scored = df_scored.withColumn(
    'risk_score',
    F.least(
        F.col('score_dpd') +
        F.col('score_npa') +
        F.col('score_kyc') +
        F.col('score_declines') +
        F.col('score_inactivity') +
        F.col('score_risk_category'),
        F.lit(100)    # Cap at 100
    ).cast(IntegerType())
)

# ── Risk Label ─────────────────────────────────────────────────────────────
df_scored = df_scored.withColumn(
    'risk_label',
    F.when(F.col('risk_score') >= 76, 'Very High Risk')
     .when(F.col('risk_score') >= 51, 'High Risk')
     .when(F.col('risk_score') >= 26, 'Medium Risk')
     .otherwise('Low Risk')
)

print('Risk scores calculated!')
df_scored.select('customer_id','risk_score','risk_label',
                 'score_dpd','score_npa','score_kyc',
                 'score_declines','score_inactivity').show(10)

## Step 5: Save to Gold and Validate

In [ ]:
# Select only the columns Power BI needs
df_gold_scores = df_scored.select(
    'customer_id',
    'segment',
    'age_group',
    'state',
    'risk_category',       # Original source system category
    'risk_score',          # Our calculated score (0-100)
    'risk_label',          # Low / Medium / High / Very High
    'score_dpd',           # Score breakdown — useful for explaining to stakeholders
    'score_npa',
    'score_kyc',
    'score_declines',
    'score_inactivity',
    'max_dpd',
    'npa_loan_count',
    'total_outstanding',
    'txn_count_30d',
    'declined_count_30d',
).withColumn('score_date', F.lit(SCORE_DATE))

df_gold_scores.write.format('delta').mode('overwrite').saveAsTable('gold_credit_risk_scores')
print(f'✅ gold_credit_risk_scores saved: {df_gold_scores.count():,} customers')

In [ ]:
# Distribution of risk segments — sanity check
print('Risk Score Distribution:')
print('=' * 40)

risk_dist = (
    df_gold_scores
    .groupBy('risk_label')
    .agg(
        F.count('*').alias('customer_count'),
        F.round(F.avg('risk_score'), 1).alias('avg_score'),
        F.round(F.avg('total_outstanding'), 0).alias('avg_outstanding'),
    )
    .orderBy(F.col('avg_score').desc())
)

risk_dist.show()

total = df_gold_scores.count()
for row in risk_dist.collect():
    pct = round(row['customer_count'] / total * 100, 1)
    bar = '█' * int(pct / 3)
    print(f'{row["risk_label"]:15s}  {pct:5.1f}%  {bar}')

print()
print('✅ All 4 notebooks complete!')
print('📊 Connect Power BI to the bankguard Lakehouse SQL endpoint')
print('   Tables to use: gold_monthly_transactions, gold_customer_summary,')
print('                  gold_loan_portfolio, gold_merchant_analysis, gold_credit_risk_scores')